# Fashion-MNIST TC6 Spatial Extension (S0, S1, S3, S6-B)

Notebook này chỉ đọc 15 source runs B0/S0 đã hoàn tất, kiểm tra assignments và sinh ba encoding mới: S1 (grid 2×2), S3 (horizontal stripes 2–3–2) và S6-B (ABOVE/LEFT_OF ở Manhattan distance 1–2). Không chạy TC1–TC5, EDA2 hoặc Pilot Eclat.

Mặc định notebook dùng `configs/tc6_spatial_extension_full.toml` và tạo đủ 45 output. Để thử isolated smoke trước, đặt `ENV["FASHION_TC6_CONFIG"] = "configs/tc6_spatial_extension_smoke.toml"` trước khi chạy cell 00; smoke ghi 100 ảnh của một source vào root riêng.

## 00 — Environment & Config

Tìm root repo, activate project và đọc config TC6. Không gọi instantiate hoặc bước huấn luyện.

In [ ]:
using Pkg
using TOML

function tc6_find_repo_root(start = pwd())
    current = abspath(start)
    while true
        isfile(joinpath(current, "Project.toml")) && return current
        parent = dirname(current)
        parent == current && error("Không tìm thấy Project.toml từ $start")
        current = parent
    end
end

const TC6_REPO_ROOT = tc6_find_repo_root()
cd(TC6_REPO_ROOT)
Pkg.activate(TC6_REPO_ROOT)
using EclatFIM
const TC6 = EclatFIM.FashionTCSpatial
const TC6_CONFIG_PATH = get(ENV, "FASHION_TC6_CONFIG", joinpath(TC6_REPO_ROOT, "configs", "tc6_spatial_extension_full.toml"))
tc6_context = TC6.load_context(TC6_CONFIG_PATH; repo_root=TC6_REPO_ROOT)
println("Repo: ", TC6_REPO_ROOT)
println("Julia: ", VERSION)
println("Config: ", relpath(TC6_CONFIG_PATH, TC6_REPO_ROOT))
println("Profile: ", tc6_context.profile)
println("Source: ", tc6_context.source_root)
println("Output: ", tc6_context.output_root)

## 01 — Source Discovery

Khám phá theo run ID/metadata và đối chiếu đủ 15 cặp K/seed canonical; không đọc summary có thể chứa duplicate để xác định codebook.

In [ ]:
tc6_sources = TC6.discover_sources(tc6_context)
@assert length(tc6_sources) == 15
[(run_id=s.run_id, k=s.k, seed=s.seed) for s in tc6_sources]

## 02 — Source Preflight

Đọc từng source một lần để xác nhận checkpoint/integrity, 70.000 S0 transactions, mapping, tọa độ retained patches và projection về S0. Dữ liệu mỗi source được giải phóng trước khi sang source kế tiếp.

In [ ]:
tc6_source_manifest = TC6.preflight_sources!(tc6_context)
@assert length(tc6_source_manifest) == 15
println("Validated source runs: ", length(tc6_source_manifest))
println("Common TID/image mapping SHA-256: ", tc6_context.mapping_hash)

## 03 — Load & Normalize Patch Assignments

Preflight chuẩn hóa tọa độ thành lưới 7×7 1-based, giữ patch_id gốc và kiểm từng ảnh với S0. Bảng dưới đây là manifest nguồn và số retained assignments; không nạp đồng thời 15 source datasets.

In [ ]:
[(run_id=row["source_run_id"], k=row["k"], seed=row["seed"],
  retained_patches=row["retained_patch_count"], s0_empty=row["s0_empty_count"],
  fit_status=row["fit_status"], converged=row["converged"])
 for row in tc6_source_manifest]

## 04 — Encoder Functions & Unit Checks

Kiểm tra chia vùng 3+4, stripes 2–3–2, round-trip item IDs, 226 neighbor pairs/72 diagonals, hướng quan hệ, boundary distance, same-word, dedup, input đảo thứ tự và trường hợp rỗng.

In [ ]:
tc6_unit_checks = TC6.run_unit_checks!()
@assert all(==("PASS"), values(tc6_unit_checks))
tc6_unit_checks

## 05 — S1 Encode → Export → Read-back

Mỗi output có 70.000 transactions ở full mode, mapping observed items, TID mapping nguồn, config/provenance/stats, integrity report và checkpoint. S1 projection được so với S0 cho mọi ảnh.

In [ ]:
tc6_s1_results = TC6.run_variant!(tc6_context, "S1")
@assert length(tc6_s1_results) == (tc6_context.profile == "full" ? 15 : 1)
tc6_s1_results

## 06 — S3 Encode → Export → Read-back

Dùng stripes TOP/MIDDLE/BOTTOM theo 2–3–2 hàng. Projection giải mã về S0 được kiểm tra đầy đủ theo từng ảnh.

In [ ]:
tc6_s3_results = TC6.run_variant!(tc6_context, "S3")
@assert length(tc6_s3_results) == (tc6_context.profile == "full" ? 15 : 1)
tc6_s3_results

## 07 — S6-B Encode → Export → Read-back

Chỉ sinh ABOVE và LEFT_OF giữa retained patch pairs khác vị trí, Manhattan distance 1–2. Diagonal có thể sinh cả hai quan hệ; mỗi observed relation được kiểm witness trước khi checkpoint hoàn tất.

In [ ]:
tc6_s6b_results = TC6.run_variant!(tc6_context, "S6-B")
@assert length(tc6_s6b_results) == (tc6_context.profile == "full" ? 15 : 1)
tc6_s6b_results

## 08 — Cross-variant & Source Integrity

Đọc lại trạng thái/checksum mọi output đã hoàn tất và so checksum artifacts nguồn với manifest preflight; nguồn S0 vẫn được bảo toàn.

In [ ]:
tc6_integrity = TC6.validate_global_integrity!(tc6_context)
tc6_integrity

## 09 — Global Summaries & EDA2 Manifest

Tạo summary runs và manifest. Full profile phải có 45 output PASS cùng 60 database references; smoke manifest được đánh dấu `eda2_ready=false` và không dùng làm input EDA2.

In [ ]:
tc6_final = TC6.finalize_outputs!(tc6_context)
println("Summary: ", joinpath(tc6_context.output_root, "spatial_extension_run_summary.toml"))
println("Manifest: ", joinpath(tc6_context.output_root, tc6_context.profile == "full" ? "eda2_database_manifest.toml" : "eda2_database_manifest_smoke.toml"))
println("Manifest database count: ", tc6_final.manifest["database_count"])
println("EDA2 ready: ", tc6_final.manifest["eda2_ready"])

## 10 — Execution Summary / Handover

Tóm tắt run count, profile, trạng thái và output root. Cell này không chạy EDA2, Pilot Eclat, permutation null model hay chọn K/seed.

In [ ]:
tc6_summary = TOML.parsefile(joinpath(tc6_context.output_root, "spatial_extension_run_summary.toml"))
println("Profile: ", tc6_summary["profile"])
println("Completed spatial runs: ", tc6_summary["completed_new_runs"], "/", tc6_summary["expected_new_runs"])
println("Output root: ", tc6_context.output_root)
[(run_id=row["run_id"], variant=row["variant"], k=row["k"], seed=row["seed"],
  status=row["status"], records=row["transaction_count"], integrity=row["integrity"])
 for row in tc6_summary["runs"]]